# XCT-IDS revision pipeline — version 2

**Stage 1 — address the reviewer comments.** A clean rerun of the paper's design: deduplicate, split, augment the
paper's six minority classes with CTGAN, filter with SHAP, train the multi-task Transformer, compare with matched
filters and baselines over 5 seeds. It also adds the correlation statistics (R1-7), redraws every figure, saves
generator checkpoints (R1-8) and writes `reviewer_evidence.md` with every number filled in.

**Stage 2 — try to improve the method (acceptance chances).** A fixed generator (CTGAN with discrete-valued
columns declared discrete + log transform, `ctganD`), TVAE as a second generator, and augmentation applied to XGBoost.
The success criteria are fixed in Cell 1 **before** running and checked automatically in `stage2_verdict.json`.

**Run on Kaggle:** GPU T4 x1, Internet On, UNSW-NB15 attached (swapped file names are handled).
1. `RUN_MODE = "smoke"` first (~15 min). 2. `RUN_MODE = "full"`, then **Save Version → Save & Run All**.
Full run with both stages ≈ 5–6 h; Stage 1 alone ≈ 3 h (set `RUN_STAGE2 = False`).


In [5]:
# %% Cell 1 — configuration
RUN_MODE = "smoke"          # "smoke" = fast end-to-end check; "full" = paper run
RUN_STAGE1 = True           # reviewer-comment package
RUN_STAGE2 = True           # improved generators + XGBoost augmentation + pre-registered verdict

RUN_SENSITIVITY = True      # Stage 1: target ratio x removal fraction on VALIDATION only (R2-3)
RUN_DIAGNOSTICS = True      # realism / memorisation / coverage for every generator (R2-4)
RUN_STATS = True            # correlation + statistical similarity (R1-7)
RUN_EXPLAIN = True          # SHAP on the final classifiers + fidelity + stability (R2-5)
RUN_FIGURES = True          # all paper figures from saved results (R1-8)

SEEDS = [0, 1, 2, 3, 4]
SPLIT_SEED = 42

STAGE1_VARIANTS = ["real_only", "ctgan_nofilter", "ctgan_shap", "ctgan_shaponly", "ctgan_conf",
                   "ctgan_random", "smote", "adasyn"]
STAGE2_VARIANTS = ["ctganD_nofilter", "ctganD_shap", "ctganD_random",
                   "tvae_nofilter", "tvae_shap", "tvae_random"]
XGB_STAGE1 = ["real_only"]
XGB_STAGE2 = ["ctgan_nofilter", "ctgan_shap", "ctgan_random", "ctganD_nofilter", "ctganD_shap", "ctganD_random",
              "tvae_nofilter", "tvae_shap", "tvae_random"]
EXPLAIN_VARIANTS = ["ctgan_shap", "real_only"]

# Augmentation policy (paper's design)
MINORITY_RULE = 0.25        # used only if MINORITY_CLASSES is None
MINORITY_CLASSES = ["DoS", "Reconnaissance", "Analysis", "Backdoor", "Shellcode", "Worms"]
TARGET_RATIO = 0.40
POOL_RATIO = 0.60
REMOVAL_FRAC = 0.10
QUALITY_ALPHA = 0.50
VAL_FRAC = 0.20

# Generators
CTGAN_EPOCHS = 150
TVAE_EPOCHS = 300
GEN_BATCH = 500
CTGAN_PAC = 10
DISCRETE_MAX_UNIQUE = 32    # ctganD/tvae: numeric columns with <= 32 distinct values in the class are discrete
SAVE_GENERATOR_SEEDS = [0]  # generator checkpoints saved for these seeds (R1-8)

# Discriminator
DISC_FOLDS, DISC_TREES = 5, 300

# Transformer
D_MODEL, N_HEADS, N_LAYERS, D_FF, DROPOUT = 64, 8, 3, 128, 0.10
BATCH, LR, WEIGHT_DECAY = 1024, 1e-3, 1e-5
MAX_EPOCHS, PATIENCE = 40, 5
LAMBDA_MULTI = 1.0
BINARY_THRESHOLD = 0.5

# XGBoost
XGB_TREES, XGB_DEPTH = 2000, 8

SENS_RATIOS = [0.20, 0.40, 0.60]
SENS_REMOVALS = [0.0, 0.10, 0.20]
DIAG_MAX = 5000
EXPLAIN_N, EXPLAIN_BG = 200, 100
N_CORR_FEATURES = 20

# ---- Pre-registered Stage 2 success criteria (fixed BEFORE running; do not edit after seeing results) ----
CRITERIA = {
    "max_discriminator_auc": 0.90,          # C1: synthetic data not trivially separable
    "max_dcr_ratio": 2.0,                   # C2: median synthetic->real distance <= 2x real holdout->real distance
    "require_ci_above_zero_vs": ["real_only", "random"],   # C3: proposed beats both, 95% CI of macro-F1 diff > 0
    "metric": "multi_macro_f1",
    "proposed_selected_on": "mean validation score of <generator>_shap",   # test set is not used for selection
}

if RUN_MODE == "smoke":
    SEEDS = [0, 1]
    SUBSAMPLE = 0.10
    CTGAN_EPOCHS, TVAE_EPOCHS = 3, 3
    DISC_TREES, XGB_TREES = 50, 100
    MAX_EPOCHS, PATIENCE = 2, 1
    SENS_RATIOS, SENS_REMOVALS = [0.40], [0.0, 0.10]
    DIAG_MAX = 500
    EXPLAIN_N, EXPLAIN_BG = 20, 20
else:
    SUBSAMPLE = None

VARIANTS = (STAGE1_VARIANTS if RUN_STAGE1 else []) + (STAGE2_VARIANTS if RUN_STAGE2 else [])
if RUN_STAGE2 and "real_only" not in VARIANTS:
    VARIANTS = ["real_only"] + VARIANTS                    # needed for the Stage 2 comparison
XGB_VARIANTS = (XGB_STAGE1 if RUN_STAGE1 else []) + (XGB_STAGE2 if RUN_STAGE2 else [])
if RUN_STAGE2 and "real_only" not in XGB_VARIANTS:
    XGB_VARIANTS = ["real_only"] + XGB_VARIANTS
GENERATORS = sorted({v.split("_")[0] for v in VARIANTS + XGB_VARIANTS if v.split("_")[0] in ("ctgan", "ctganD", "tvae")})

from pathlib import Path
OUTPUT_DIR = Path("/kaggle/working/xct_ids_v2" if Path("/kaggle/working").exists() else "./xct_ids_v2") / RUN_MODE
for sub in ["tables", "synthetic", "scores", "preds", "models", "generators", "logs", "figures"]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR, "| generators:", GENERATORS)
print("Transformer variants:", VARIANTS)
print("XGBoost variants:", XGB_VARIANTS)


Output: /kaggle/working/xct_ids_v2/smoke | generators: ['ctgan', 'ctganD', 'tvae']
Transformer variants: ['real_only', 'ctgan_nofilter', 'ctgan_shap', 'ctgan_shaponly', 'ctgan_conf', 'ctgan_random', 'smote', 'adasyn', 'ctganD_nofilter', 'ctganD_shap', 'ctganD_random', 'tvae_nofilter', 'tvae_shap', 'tvae_random']
XGBoost variants: ['real_only', 'ctgan_nofilter', 'ctgan_shap', 'ctgan_random', 'ctganD_nofilter', 'ctganD_shap', 'ctganD_random', 'tvae_nofilter', 'tvae_shap', 'tvae_random']


In [6]:
# %% Cell 2 — imports and helpers
import subprocess, sys
try:
    import ctgan  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ctgan"], check=True)

import copy, json, random, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score, matthews_corrcoef,
                             roc_auc_score, average_precision_score, confusion_matrix)
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True
print("Device:", DEVICE, torch.cuda.get_device_name(0) if USE_AMP else "")

CLASSES = ["Normal", "Analysis", "Backdoor", "DoS", "Exploits", "Fuzzers",
           "Generic", "Reconnaissance", "Shellcode", "Worms"]
CLS_IDX = {c: i for i, c in enumerate(CLASSES)}
CAT_COLS = ["proto", "service", "state"]


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if getattr(o, "size", 0) == 1
                  else (o.tolist() if hasattr(o, "tolist") else str(o)))


def append_row(path, row):
    df = pd.DataFrame([row])
    df.to_csv(path, mode="a", header=not Path(path).exists(), index=False)


class Timer:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a):
        self.s = time.time() - self.t
        print(f"[{self.name}] {self.s/60:.1f} min")
        append_row(OUTPUT_DIR / "logs" / "timings.csv", {"step": self.name, "seconds": round(self.s, 1)})


LEDGER = []  # per-class counts at every stage (answers R1-1 / R1-2)


def ledger(stage, series_or_df, note=""):
    s = series_or_df["attack_cat"] if isinstance(series_or_df, pd.DataFrame) else series_or_df
    counts = pd.Series(s).value_counts().reindex(CLASSES, fill_value=0)
    row = {"stage": stage, **counts.to_dict(), "total": int(counts.sum()), "note": note}
    LEDGER.append(row)
    pd.DataFrame(LEDGER).to_csv(OUTPUT_DIR / "tables" / "ledger_counts.csv", index=False)


Device: cuda Tesla T4


In [7]:
# %% Cell 3 — load, clean, deduplicate (official test partition is never modified)
def find_unsw_files():
    root = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(".")
    tr_files = sorted(root.rglob("UNSW_NB15_training-set.csv"))
    te_files = sorted(root.rglob("UNSW_NB15_testing-set.csv"))
    if not tr_files or not te_files:
        raise FileNotFoundError("Could not find both UNSW_NB15_training-set.csv and UNSW_NB15_testing-set.csv")
    files = [tr_files[0], te_files[0]]   # if several copies are attached, the first of each is used
    a, b = pd.read_csv(files[0]), pd.read_csv(files[1])
    # The official training partition has 175,341 rows; take the larger file as train.
    return (a, b, files[0], files[1]) if len(a) > len(b) else (b, a, files[1], files[0])


def clean(df):
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    df["attack_cat"] = df["attack_cat"].fillna("Normal").astype(str).str.strip()
    df["attack_cat"] = df["attack_cat"].replace({"": "Normal", "Backdoors": "Backdoor"})
    df["attack_cat"] = df["attack_cat"].str.lower().map({c.lower(): c for c in CLASSES})
    if df["attack_cat"].isna().any():
        raise ValueError("Unrecognised attack_cat values")
    label_mismatch = int(((df["attack_cat"] == "Normal") != (df["label"] == 0)).sum())
    df["_row"] = np.arange(len(df))  # position in the official CSV, for split indices
    df = df.drop(columns=["id"], errors="ignore")
    for c in CAT_COLS:
        df[c] = df[c].astype(str).str.strip()
    return df, label_mismatch


raw_train, raw_test, train_path, test_path = find_unsw_files()
train_full, mm_tr = clean(raw_train)
test_df, mm_te = clean(raw_test)
NUM_COLS = [c for c in train_full.columns if c not in CAT_COLS + ["attack_cat", "label", "_row"]]
FEATURES = NUM_COLS + CAT_COLS
for c in NUM_COLS:
    train_full[c] = pd.to_numeric(train_full[c], errors="coerce").fillna(0.0)
    test_df[c] = pd.to_numeric(test_df[c], errors="coerce").fillna(0.0)
print(f"Train file: {train_path} -> {len(train_full):,} rows | Test file: {test_path} -> {len(test_df):,} rows")
print(f"{len(NUM_COLS)} numerical + {len(CAT_COLS)} categorical = {len(FEATURES)} features")
ledger("official_train", train_full)
ledger("official_test", test_df)


def row_hash(df):
    return pd.util.hash_pandas_object(df[FEATURES], index=False)


# Exact duplicates (same features AND same class) inside the official training partition
n_before = len(train_full)
train_full = train_full.loc[~train_full.duplicated(subset=FEATURES + ["attack_cat"])].reset_index(drop=True)
n_dupes = n_before - len(train_full)
h_tr = row_hash(train_full)
label_conflicts = int(train_full.assign(_h=h_tr.values).duplicated("_h", keep=False).sum())
test_in_train = row_hash(test_df).isin(set(h_tr.values))
data_audit = {
    "train_rows_official": n_before, "train_exact_duplicates_removed": n_dupes,
    "train_rows_after_dedup": len(train_full),
    "train_rows_with_same_features_different_class": label_conflicts,
    "test_rows": len(test_df), "test_exact_duplicate_rows": int(test_df.duplicated(subset=FEATURES).sum()),
    "test_rows_whose_features_appear_in_train": int(test_in_train.sum()),
    "label_vs_attack_cat_mismatch_train": mm_tr, "label_vs_attack_cat_mismatch_test": mm_te,
}
print(json.dumps(data_audit, indent=2))
save_json(data_audit, OUTPUT_DIR / "tables" / "data_audit.json")
ledger("train_after_dedup", train_full, f"{n_dupes} exact duplicates removed")

if SUBSAMPLE:
    def strat_sample(df, floor):
        parts = [g.sample(max(int(len(g) * SUBSAMPLE), min(len(g), floor)), random_state=0)
                 for _, g in df.groupby("attack_cat")]
        return pd.concat(parts).reset_index(drop=True)
    train_full, test_df = strat_sample(train_full, 30), strat_sample(test_df, 20)
    test_in_train = row_hash(test_df).isin(set(row_hash(train_full).values))
    print("SMOKE subsample:", len(train_full), "train /", len(test_df), "test")


Train file: /kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW_NB15_testing-set.csv -> 175,341 rows | Test file: /kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW_NB15_training-set.csv -> 82,332 rows
39 numerical + 3 categorical = 42 features
{
  "train_rows_official": 175341,
  "train_exact_duplicates_removed": 67601,
  "train_rows_after_dedup": 107740,
  "train_rows_with_same_features_different_class": 8472,
  "test_rows": 82332,
  "test_exact_duplicate_rows": 28386,
  "test_rows_whose_features_appear_in_train": 8541,
  "label_vs_attack_cat_mismatch_train": 0,
  "label_vs_attack_cat_mismatch_test": 0
}
SMOKE subsample: 10789 train / 8245 test


In [8]:
# %% Cell 4 — train / validation split (BEFORE any augmentation or fitting)
tr_idx, va_idx = train_test_split(np.arange(len(train_full)), test_size=VAL_FRAC,
                                  random_state=SPLIT_SEED, stratify=train_full["attack_cat"])
train_df = train_full.iloc[tr_idx].reset_index(drop=True)
val_df = train_full.iloc[va_idx].reset_index(drop=True)
pd.DataFrame({"official_row": train_df["_row"]}).to_csv(OUTPUT_DIR / "tables" / "split_train_rows.csv", index=False)
pd.DataFrame({"official_row": val_df["_row"]}).to_csv(OUTPUT_DIR / "tables" / "split_val_rows.csv", index=False)
ledger("train_split", train_df)
ledger("validation_split", val_df)

counts = train_df["attack_cat"].value_counts().reindex(CLASSES, fill_value=0)
LARGEST = int(counts.max())
RULE_MINORITY = [c for c in CLASSES if counts[c] < MINORITY_RULE * LARGEST]
MINORITY = list(MINORITY_CLASSES) if MINORITY_CLASSES else RULE_MINORITY
print("Minority by 25 % rule:", RULE_MINORITY, "| augmented:", MINORITY)


def n_needed(cls, ratio):
    return max(0, int(ratio * LARGEST) - int(counts[cls]))


policy = pd.DataFrame([{"class": c, "train_split_count": int(counts[c]),
                        "imbalance_ratio": round(LARGEST / max(counts[c], 1), 2),
                        "minority": c in MINORITY,
                        "target": int(TARGET_RATIO * LARGEST) if c in MINORITY else int(counts[c]),
                        "synthetic_required": n_needed(c, TARGET_RATIO) if c in MINORITY else 0}
                       for c in CLASSES])
policy.to_csv(OUTPUT_DIR / "tables" / "augmentation_policy.csv", index=False)
print(policy.to_string(index=False))


Minority by 25 % rule: ['Analysis', 'Backdoor', 'DoS', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms'] | augmented: ['DoS', 'Reconnaissance', 'Analysis', 'Backdoor', 'Shellcode', 'Worms']
         class  train_split_count  imbalance_ratio  minority  target  synthetic_required
        Normal               4151             1.00     False    4151                   0
      Analysis                127            32.69      True    1660                1533
      Backdoor                123            33.75      True    1660                1537
           DoS                304            13.65      True    1660                1356
      Exploits               1587             2.62     False    1587                   0
       Fuzzers               1292             3.21     False    1292                   0
       Generic                334            12.43     False     334                   0
Reconnaissance                602             6.90      True    1660                1058
     She

In [9]:
# %% Cell 5 — preprocessing fitted on the REAL train split only
class Prep:
    """Numerical: signed log1p then z-score. Categorical: integer tokens, 0 = unseen."""

    def fit(self, df):
        z = self._slog(df[NUM_COLS].to_numpy(np.float64))
        self.mu, self.sd = z.mean(0), z.std(0) + 1e-6
        self.vocab = {c: {v: i + 1 for i, v in enumerate(sorted(df[c].unique()))} for c in CAT_COLS}
        self.cards = [len(self.vocab[c]) + 1 for c in CAT_COLS]
        return self

    @staticmethod
    def _slog(x):
        return np.sign(x) * np.log1p(np.abs(x))

    def num(self, df):
        z = (self._slog(df[NUM_COLS].to_numpy(np.float64)) - self.mu) / self.sd
        return np.clip(z, -10, 10).astype(np.float32)

    def cat(self, df):
        return np.stack([df[c].map(self.vocab[c]).fillna(0).to_numpy(np.int64) for c in CAT_COLS], 1)

    def onehot(self, xc):
        return np.concatenate([np.eye(k, dtype=np.float32)[xc[:, j]] for j, k in enumerate(self.cards)], 1)

    def labels(self, df):
        ym = df["attack_cat"].map(CLS_IDX).to_numpy(np.int64)
        return (ym != CLS_IDX["Normal"]).astype(np.float32), ym


prep = Prep().fit(train_df)
save_json({"numerical": NUM_COLS, "categorical": CAT_COLS, "cardinalities": prep.cards,
           "numerical_transform": "sign(x)*log1p(|x|), then z-score (train split statistics), clipped to [-10, 10]",
           "categorical_transform": "integer token per category seen in train split; 0 = unseen",
           "tokens": f"{len(NUM_COLS)} numerical + {len(CAT_COLS)} categorical feature tokens + 1 CLS token"},
          OUTPUT_DIR / "tables" / "preprocessing.json")
INT_COLS = [c for c in NUM_COLS if np.allclose(train_df[c] % 1, 0)]  # integer-valued in real data


In [10]:
# %% Cell 6 — synthetic data generators (cached per generator / seed / class)
#   ctgan  : Stage 1, as in the paper (only proto/service/state discrete; negatives clipped, integers rounded)
#   ctganD : Stage 2, CTGAN with low-cardinality numeric columns discrete + log1p on heavy-tailed columns
#   tvae   : Stage 2, TVAE with the same column treatment as ctganD
def _gpu_kw(cls, **kw):
    try:
        return cls(enable_gpu=torch.cuda.is_available(), **kw)
    except TypeError:
        return cls(cuda=torch.cuda.is_available(), **kw)


def make_generator(gen):
    from ctgan import CTGAN, TVAE
    if gen in ("ctgan", "ctganD"):
        return _gpu_kw(CTGAN, epochs=CTGAN_EPOCHS, batch_size=GEN_BATCH, pac=CTGAN_PAC,
                       generator_dim=(256, 256), discriminator_dim=(256, 256), verbose=False)
    if gen == "tvae":
        return _gpu_kw(TVAE, epochs=TVAE_EPOCHS, batch_size=GEN_BATCH)
    raise ValueError(gen)


def violation_rates(syn, real):
    neg_cols = [c for c in NUM_COLS if real[c].min() >= 0]
    return {"non_integer_rate": np.mean([(syn[c] % 1 != 0).mean() for c in INT_COLS]) if INT_COLS else 0.0,
            "negative_rate": np.mean([(syn[c] < 0).mean() for c in neg_cols]) if neg_cols else 0.0,
            "above_real_max_rate": np.mean([(syn[c] > real[c].max()).mean() for c in NUM_COLS]),
            "below_real_min_rate": np.mean([(syn[c] < real[c].min()).mean() for c in NUM_COLS])}


def column_plan(real):
    disc_num = [c for c in NUM_COLS if real[c].nunique() <= DISCRETE_MAX_UNIQUE]
    cont = [c for c in NUM_COLS if c not in disc_num]
    log_cols = [c for c in cont if real[c].min() >= 0 and real[c].max() > 100]
    return disc_num, cont, log_cols


def fit_sample(gen, real, n, seed, cls):
    set_seed(seed)
    model = make_generator(gen)
    if hasattr(model, "set_random_state"):
        model.set_random_state(seed)
    if gen == "ctgan":
        model.fit(real, discrete_columns=CAT_COLS)
        syn = model.sample(n)[FEATURES]
        for c in NUM_COLS:
            syn[c] = pd.to_numeric(syn[c], errors="coerce").fillna(real[c].median())
        viol = violation_rates(syn, real)
        for c in [c for c in NUM_COLS if real[c].min() >= 0]:
            syn[c] = syn[c].clip(lower=0)
        plan = {"discrete_numeric": [], "log_columns": []}
    else:
        disc_num, cont, log_cols = column_plan(real)
        X = real.copy()
        for c in disc_num:
            X[c] = X[c].astype(str)
        X[log_cols] = np.log1p(X[log_cols])
        model.fit(X, discrete_columns=CAT_COLS + disc_num)
        syn = model.sample(n)[FEATURES]
        for c in disc_num:
            syn[c] = pd.to_numeric(syn[c].astype(str), errors="coerce")
        syn[log_cols] = np.expm1(syn[log_cols].astype(float))
        for c in NUM_COLS:
            syn[c] = pd.to_numeric(syn[c], errors="coerce").fillna(real[c].median())
        viol = violation_rates(syn, real)
        for c in NUM_COLS:                                   # repair to the observed class range
            syn[c] = syn[c].clip(real[c].min(), real[c].max())
        plan = {"discrete_numeric": disc_num, "log_columns": log_cols}
    for c in INT_COLS:
        syn[c] = syn[c].round()
    if seed in SAVE_GENERATOR_SEEDS and hasattr(model, "save"):
        try:
            model.save(str(OUTPUT_DIR / "generators" / f"{gen}_seed{seed}_{cls}.pkl"))
        except Exception as err:
            print("  generator checkpoint not saved:", err)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return syn, viol, plan


def get_pool(gen, seed):
    pool = {}
    for cls in MINORITY:
        path = OUTPUT_DIR / "synthetic" / f"{gen}_seed{seed}_{cls}.pkl"
        if path.exists():
            pool[cls] = pd.read_pickle(path); continue
        n = n_needed(cls, POOL_RATIO)
        if n == 0:
            continue
        real = train_df.loc[train_df["attack_cat"] == cls, FEATURES].reset_index(drop=True)
        with Timer(f"{gen} seed{seed} {cls} (n_real={len(real)}, n_gen={n})"):
            syn, viol, plan = fit_sample(gen, real, n, seed, cls)
        append_row(OUTPUT_DIR / "tables" / "generator_domain_violations.csv",
                   {"generator": gen, "seed": seed, "class": cls, "n_real": len(real), "n_generated": n,
                    "n_discrete_numeric": len(plan["discrete_numeric"]), "n_log_columns": len(plan["log_columns"]),
                    **viol})
        if seed == SEEDS[0]:
            save_json(plan, OUTPUT_DIR / "tables" / f"generator_column_plan_{gen}_{cls}.json")
        syn["attack_cat"] = cls
        syn.to_pickle(path)
        pool[cls] = syn
    return pool


def take(pool, ratio):
    parts = [pool[c].iloc[:n_needed(c, ratio)] for c in MINORITY if c in pool]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=FEATURES + ["attack_cat"])


In [11]:
# %% Cell 7 — cross-fitted real-vs-synthetic discriminator + TreeSHAP quality scores (R1-6)
import xgboost as xgb


def disc_features(df):
    xn, xc = prep.num(df), prep.cat(df)
    cls = np.eye(len(MINORITY), dtype=np.float32)[df["attack_cat"].map({c: i for i, c in enumerate(MINORITY)})]
    return np.concatenate([xn, prep.onehot(xc), cls], 1)


def disc_feature_groups():
    groups = list(NUM_COLS)
    for c, k in zip(CAT_COLS, prep.cards):
        groups += [c] * k
    return np.array(groups + ["class_indicator"] * len(MINORITY))


SCORED = {}


def score_synthetic(gen, seed, ratio=TARGET_RATIO):
    key = (gen, seed, ratio)
    if key in SCORED:
        return SCORED[key]
    path = OUTPUT_DIR / "scores" / f"{gen}_r{ratio}_seed{seed}.pkl"
    if path.exists():
        SCORED[key] = (pd.read_pickle(path), json.load(open(path.with_suffix(".json"))))
        return SCORED[key]
    syn = take(get_pool(gen, seed), ratio)
    real = train_df[train_df["attack_cat"].isin(MINORITY)].reset_index(drop=True)
    X = np.concatenate([disc_features(real), disc_features(syn)])
    y = np.r_[np.zeros(len(real)), np.ones(len(syn))]
    strat = y * 100 + np.r_[real["attack_cat"].map(CLS_IDX), syn["attack_cat"].map(CLS_IDX)]
    margin = np.zeros(len(y))
    contrib = np.zeros((len(syn), X.shape[1]), dtype=np.float32)
    w_pos = len(real) / len(syn)
    with Timer(f"discriminator {gen} r{ratio} seed{seed}"):
        for k, (a, b) in enumerate(StratifiedKFold(DISC_FOLDS, shuffle=True, random_state=seed).split(X, strat)):
            clf = xgb.XGBClassifier(n_estimators=DISC_TREES, max_depth=6, learning_rate=0.05, subsample=0.8,
                                    colsample_bytree=0.8, tree_method="hist",
                                    device="cuda" if USE_AMP else "cpu", random_state=seed + k)
            clf.fit(X[a], y[a], sample_weight=np.where(y[a] == 1, w_pos, 1.0))
            booster = clf.get_booster()
            margin[b] = booster.predict(xgb.DMatrix(X[b]), output_margin=True)
            sb = b[b >= len(real)]
            if len(sb):
                contrib[sb - len(real)] = booster.predict(xgb.DMatrix(X[sb]), pred_contribs=True)[:, :-1]
    syn_margin = margin[len(real):]
    p = 1 / (1 + np.exp(-syn_margin))
    dev = np.maximum(contrib, 0).sum(1) / (np.abs(contrib).sum(1) + 1e-12)
    out = syn.copy()
    out["disc_margin"], out["p_synthetic"], out["shap_deviation"] = syn_margin, p, dev
    out["q_shap"] = 1 - (QUALITY_ALPHA * p + (1 - QUALITY_ALPHA) * dev)
    out["q_shaponly"] = 1 - dev
    imp = pd.Series(np.abs(contrib).mean(0)).groupby(disc_feature_groups()).sum().sort_values(ascending=False)
    by_class = {}
    for c in MINORITY:
        rm, sm = (real.attack_cat == c).to_numpy(), (syn.attack_cat == c).to_numpy()
        if sm.any() and rm.any():
            by_class[c] = float(roc_auc_score(np.r_[np.zeros(rm.sum()), np.ones(sm.sum())],
                                              np.r_[margin[:len(real)][rm], syn_margin[sm]]))
    info = {"generator": gen, "seed": seed, "ratio": ratio, "oof_auc_all": float(roc_auc_score(y, margin)),
            "oof_auc_by_class": by_class, "mean_p_synthetic": float(p.mean()),
            "shap_scale": "log-odds (XGBoost pred_contribs, exact TreeSHAP)",
            "global_importance_42_features": imp.round(5).to_dict()}
    out.to_pickle(path)
    save_json(info, path.with_suffix(".json"))
    if ratio == TARGET_RATIO:
        append_row(OUTPUT_DIR / "tables" / "discriminator_auc.csv",
                   {"generator": gen, "seed": seed, "oof_auc_all": info["oof_auc_all"],
                    **{f"auc_{c}": v for c, v in by_class.items()}, "mean_p_synthetic": info["mean_p_synthetic"]})
    SCORED[key] = (out, info)
    return SCORED[key]


def select(scored, mode, frac, seed):
    if mode == "none" or frac <= 0:
        return scored
    rng = np.random.default_rng(seed)
    drop = []
    for _, g in scored.groupby("attack_cat"):
        k = int(round(frac * len(g)))
        if mode == "shap":
            drop += list(g.nsmallest(k, "q_shap").index)
        elif mode == "shaponly":
            drop += list(g.nsmallest(k, "q_shaponly").index)
        elif mode == "conf":
            drop += list(g.nlargest(k, "disc_margin").index)
        elif mode == "random":
            drop += list(rng.choice(g.index, k, replace=False))
        else:
            raise ValueError(mode)
    return scored.drop(index=drop)


def save_masks(gen, seed):
    scored, _ = score_synthetic(gen, seed)
    masks = scored[["attack_cat", "disc_margin", "p_synthetic", "shap_deviation", "q_shap"]].copy()
    for mode in ["shap", "shaponly", "conf", "random"]:
        masks[f"kept_{mode}"] = masks.index.isin(select(scored, mode, REMOVAL_FRAC, seed).index)
    masks.to_csv(OUTPUT_DIR / "tables" / f"retention_masks_{gen}_seed{seed}.csv.gz", index_label="synthetic_id")
    kc = masks.groupby("attack_cat")[[c for c in masks if c.startswith("kept_")]].sum()
    kc.insert(0, "generated", masks.groupby("attack_cat").size())
    kc.loc["Total"] = kc.sum()
    kc.to_csv(OUTPUT_DIR / "tables" / f"refinement_counts_{gen}_seed{seed}.csv")


In [12]:
# %% Cell 8 — multi-task FT-Transformer (shared encoder, binary head + 10-class head)
class FTTransformer(nn.Module):
    def __init__(self, n_num, cards, d=D_MODEL, heads=N_HEADS, layers=N_LAYERS, dff=D_FF, drop=DROPOUT,
                 n_classes=len(CLASSES)):
        super().__init__()
        self.num_w = nn.Parameter(torch.randn(n_num, d) * 0.02)
        self.num_b = nn.Parameter(torch.zeros(n_num, d))
        self.cat_emb = nn.Embedding(int(sum(cards)), d)
        self.register_buffer("cat_off", torch.tensor(np.r_[0, np.cumsum(cards)[:-1]], dtype=torch.long))
        self.cat_b = nn.Parameter(torch.zeros(len(cards), d))
        self.cls = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, dff, drop, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head_binary = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(drop), nn.Linear(d, 1))
        self.head_multi = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(drop), nn.Linear(d, n_classes))

    def forward(self, xn, xc):
        tn = xn.unsqueeze(-1) * self.num_w + self.num_b                      # (B, n_num, d)
        tc = self.cat_emb(xc + self.cat_off) + self.cat_b                     # (B, n_cat, d)
        h = torch.cat([self.cls.expand(xn.shape[0], -1, -1), tn, tc], 1)      # (B, 1 + 42, d)
        h = self.norm(self.encoder(h)[:, 0])                                  # CLS pooling
        return self.head_binary(h).squeeze(-1), self.head_multi(h)


def to_dev(xn, xc, yb=None, ym=None):
    t = [torch.as_tensor(xn, device=DEVICE), torch.as_tensor(xc, device=DEVICE)]
    if yb is not None:
        t += [torch.as_tensor(yb, device=DEVICE), torch.as_tensor(ym, device=DEVICE)]
    return t


@torch.no_grad()
def predict(model, xn, xc, bs=8192):
    model.eval()
    pb, pm = [], []
    for i in range(0, len(xn), bs):
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            lb, lm = model(xn[i:i + bs], xc[i:i + bs])
        pb.append(torch.sigmoid(lb.float()).cpu()); pm.append(torch.softmax(lm.float(), 1).cpu())
    return torch.cat(pb).numpy(), torch.cat(pm).numpy()


def val_score(yb, ym, pb, pm):
    """Model-selection score on VALIDATION: mean of binary F1 (attack) and multi-class macro-F1."""
    return 0.5 * f1_score(yb, pb >= BINARY_THRESHOLD, zero_division=0) + \
        0.5 * f1_score(ym, pm.argmax(1), average="macro", labels=list(range(len(CLASSES))), zero_division=0)


def train_model(train_arrays, val_arrays, seed):
    set_seed(seed)
    xn, xc, yb, ym = to_dev(*train_arrays)
    vxn, vxc = to_dev(val_arrays[0], val_arrays[1])
    model = FTTransformer(xn.shape[1], prep.cards).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    best, best_state, bad, hist = -1.0, None, 0, []
    for ep in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(len(yb), device=DEVICE, generator=gen)
        tot = 0.0
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i + BATCH]
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                lb, lm = model(xn[idx], xc[idx])
                loss = F.binary_cross_entropy_with_logits(lb.float(), yb[idx]) + \
                    LAMBDA_MULTI * F.cross_entropy(lm.float(), ym[idx])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            tot += loss.item() * len(idx)
        pb, pm = predict(model, vxn, vxc)
        s = val_score(val_arrays[2], val_arrays[3], pb, pm)
        sched.step(s)
        hist.append({"epoch": ep + 1, "train_loss": tot / len(yb), "val_score": s, "lr": opt.param_groups[0]["lr"]})
        if s > best + 1e-4:
            best, bad = s, 0
            best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist), best


In [13]:
# %% Cell 9 — metrics
def binary_metrics(y, p, thr=BINARY_THRESHOLD):
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    rec, spec = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    prec = tp / max(tp + fp, 1)
    return {"bin_acc": (tp + tn) / len(y), "bin_precision": prec, "bin_recall": rec, "bin_specificity": spec,
            "bin_f1_attack": 2 * prec * rec / max(prec + rec, 1e-12), "bin_mcc": matthews_corrcoef(y, yhat),
            "bin_roc_auc": roc_auc_score(y, p), "bin_pr_auc": average_precision_score(y, p),
            "bin_gmean": np.sqrt(rec * spec), "bin_fpr": 1 - spec, "bin_fnr": 1 - rec,
            "bin_tp": tp, "bin_tn": tn, "bin_fp": fp, "bin_fn": fn}


def multi_metrics(y, P, prefix="multi"):
    labels = list(range(len(CLASSES)))
    yhat = P.argmax(1)
    pr, rc, f1, sup = precision_recall_fscore_support(y, yhat, labels=labels, zero_division=0)
    out = {f"{prefix}_acc": accuracy_score(y, yhat), f"{prefix}_macro_precision": pr.mean(),
           f"{prefix}_macro_recall": rc.mean(), f"{prefix}_macro_f1": f1.mean(),
           f"{prefix}_weighted_f1": f1_score(y, yhat, average="weighted", labels=labels, zero_division=0)}
    try:
        out[f"{prefix}_macro_roc_auc"] = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=labels)
    except ValueError:
        out[f"{prefix}_macro_roc_auc"] = np.nan
    if prefix == "multi":
        for i, c in enumerate(CLASSES):
            out[f"P_{c}"], out[f"R_{c}"], out[f"F1_{c}"], out[f"N_{c}"] = pr[i], rc[i], f1[i], sup[i]
    return out


def evaluate(pb, pm, tb, tm):
    m = {**binary_metrics(tb, pb), **multi_metrics(tm, pm)}
    # Extra: test rows whose exact feature vector does NOT occur in the training partition
    nov = ~test_in_train.to_numpy()
    if nov.sum() > 0:
        m.update(multi_metrics(tm[nov], pm[nov], prefix="novel"))
        m["novel_n"] = int(nov.sum())
    return m


In [14]:
# %% Cell 10 — training sets for every variant
VAL_ARR = (prep.num(val_df), prep.cat(val_df), *prep.labels(val_df))
TEST_ARR = (prep.num(test_df), prep.cat(test_df), *prep.labels(test_df))
TRAIN_REAL = (prep.num(train_df), prep.cat(train_df), *prep.labels(train_df))
np.savez_compressed(OUTPUT_DIR / "preds" / "test_labels.npz", y_binary=TEST_ARR[2], y_multi=TEST_ARR[3])
MODES = {"nofilter": "none", "shap": "shap", "shaponly": "shaponly", "conf": "conf", "random": "random"}


def arrays_from(df):
    return (prep.num(df), prep.cat(df), *prep.labels(df))


def concat_arrays(a, b):
    return tuple(np.concatenate([x, y]) for x, y in zip(a, b))


def oversample(kind, ratio, seed):
    from imblearn.over_sampling import SMOTE, ADASYN
    xn, xc, _, ym = TRAIN_REAL
    X = np.concatenate([xn, prep.onehot(xc)], 1)
    strategy = {CLS_IDX[c]: int(ratio * LARGEST) for c in MINORITY if n_needed(c, ratio) > 0}
    k = int(min(5, min(counts[c] for c in MINORITY) - 1))
    sampler = SMOTE(sampling_strategy=strategy, k_neighbors=k, random_state=seed) if kind == "smote" else \
        ADASYN(sampling_strategy=strategy, n_neighbors=k, random_state=seed)
    Xr, yr = sampler.fit_resample(X, ym)
    blocks, start = [], len(NUM_COLS)
    for card in prep.cards:
        blocks.append(Xr[:, start:start + card].argmax(1)); start += card
    return (Xr[:, :len(NUM_COLS)].astype(np.float32), np.stack(blocks, 1).astype(np.int64),
            (yr != CLS_IDX["Normal"]).astype(np.float32), yr.astype(np.int64))


def build_training(variant, seed, removal=REMOVAL_FRAC, ratio=TARGET_RATIO):
    if variant == "real_only":
        return TRAIN_REAL, train_df["attack_cat"]
    if variant in ("smote", "adasyn"):
        arr = oversample(variant, ratio, seed)
        return arr, pd.Series(np.array(CLASSES)[arr[3]])
    gen, mode = variant.split("_", 1)
    scored, _ = score_synthetic(gen, seed, ratio)
    kept = select(scored, MODES[mode], removal, seed)
    return concat_arrays(TRAIN_REAL, arrays_from(kept)), \
        pd.concat([train_df["attack_cat"], kept["attack_cat"]], ignore_index=True)


def save_cm(name, seed, pm):
    pd.DataFrame(confusion_matrix(TEST_ARR[3], pm.argmax(1), labels=range(len(CLASSES))),
                 index=CLASSES, columns=CLASSES).to_csv(OUTPUT_DIR / "tables" / f"cm_{name}_s{seed}.csv")


In [15]:
# %% Cell 11 — Transformer experiments (resume-safe)
RESULTS = OUTPUT_DIR / "tables" / "results_transformer.csv"
done = set(zip(*[pd.read_csv(RESULTS)[c] for c in ["variant", "seed"]])) if RESULTS.exists() else set()
for seed in SEEDS:
    for gen in GENERATORS:
        score_synthetic(gen, seed)
        save_masks(gen, seed)
        ledger(f"{gen}_generated_seed{seed}", score_synthetic(gen, seed)[0])
    for variant in VARIANTS:
        if (variant, seed) in done:
            continue
        try:
            arr, cls_series = build_training(variant, seed)
        except (ValueError, RuntimeError) as err:
            print(f"  {variant} seed{seed} skipped: {err}")
            append_row(OUTPUT_DIR / "logs" / "skipped.csv", {"variant": variant, "seed": seed, "error": str(err)})
            continue
        ledger(f"train_set_{variant}_seed{seed}", cls_series)
        with Timer(f"transformer {variant} seed{seed}"):
            model, hist, best_val = train_model(arr, VAL_ARR, seed)
            pb, pm = predict(model, *to_dev(TEST_ARR[0], TEST_ARR[1]))
        m = evaluate(pb, pm, TEST_ARR[2], TEST_ARR[3])
        hist.to_csv(OUTPUT_DIR / "logs" / f"history_{variant}_s{seed}.csv", index=False)
        np.savez_compressed(OUTPUT_DIR / "preds" / f"{variant}_s{seed}.npz", p_binary=pb, p_multi=pm)
        torch.save(model.state_dict(), OUTPUT_DIR / "models" / f"{variant}_s{seed}.pt")
        save_cm(variant, seed, pm)
        append_row(RESULTS, {"model": "transformer", "variant": variant, "seed": seed, "n_train": len(arr[3]),
                             "epochs_run": len(hist), "best_val_score": best_val, **m})
        print(f"  {variant:16s} s{seed}: bin_acc={m['bin_acc']:.4f} multi_acc={m['multi_acc']:.4f} "
              f"macroF1={m['multi_macro_f1']:.4f}")
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


[ctgan seed0 DoS (n_real=304, n_gen=2186)] 0.1 min
[ctgan seed0 Reconnaissance (n_real=602, n_gen=1888)] 0.2 min
[ctgan seed0 Analysis (n_real=127, n_gen=2363)] 0.0 min
[ctgan seed0 Backdoor (n_real=123, n_gen=2367)] 0.0 min
[ctgan seed0 Shellcode (n_real=87, n_gen=2403)] 0.0 min
[ctgan seed0 Worms (n_real=24, n_gen=2466)] 0.0 min
[discriminator ctgan r0.4 seed0] 0.0 min
[ctganD seed0 DoS (n_real=304, n_gen=2186)] 0.0 min
[ctganD seed0 Reconnaissance (n_real=602, n_gen=1888)] 0.1 min
[ctganD seed0 Analysis (n_real=127, n_gen=2363)] 0.0 min
[ctganD seed0 Backdoor (n_real=123, n_gen=2367)] 0.0 min
[ctganD seed0 Shellcode (n_real=87, n_gen=2403)] 0.0 min
[ctganD seed0 Worms (n_real=24, n_gen=2466)] 0.0 min
[discriminator ctganD r0.4 seed0] 0.0 min
[tvae seed0 DoS (n_real=304, n_gen=2186)] 0.0 min
[tvae seed0 Reconnaissance (n_real=602, n_gen=1888)] 0.1 min
[tvae seed0 Analysis (n_real=127, n_gen=2363)] 0.0 min
[tvae seed0 Backdoor (n_real=123, n_gen=2367)] 0.0 min
[tvae seed0 Shellcode (n

In [16]:
# %% Cell 12 — XGBoost on real and augmented data (same splits, early stopping on validation)
XGB_RESULTS = OUTPUT_DIR / "tables" / "results_xgboost.csv"
done = set(zip(*[pd.read_csv(XGB_RESULTS)[c] for c in ["variant", "seed"]])) if XGB_RESULTS.exists() else set()


def xgb_matrix(arr):
    return np.concatenate([arr[0], prep.onehot(arr[1])], 1)


Xva, Xte = xgb_matrix(VAL_ARR), xgb_matrix(TEST_ARR)
for seed in SEEDS:
    for variant in XGB_VARIANTS:
        if (variant, seed) in done:
            continue
        arr, _ = build_training(variant, seed)
        Xtr = xgb_matrix(arr)
        with Timer(f"xgboost {variant} seed{seed}"):
            common = dict(n_estimators=XGB_TREES, max_depth=XGB_DEPTH, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, tree_method="hist", device="cuda" if USE_AMP else "cpu",
                          early_stopping_rounds=50, random_state=seed)
            xb = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss", **common)
            xb.fit(Xtr, arr[2], eval_set=[(Xva, VAL_ARR[2])], verbose=False)
            xm = xgb.XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", **common)
            xm.fit(Xtr, arr[3], eval_set=[(Xva, VAL_ARR[3])], verbose=False)
            pb, pm = xb.predict_proba(Xte)[:, 1], xm.predict_proba(Xte)
            vpb, vpm = xb.predict_proba(Xva)[:, 1], xm.predict_proba(Xva)
        m = evaluate(pb, pm, TEST_ARR[2], TEST_ARR[3])
        name = f"xgb-{variant}"
        np.savez_compressed(OUTPUT_DIR / "preds" / f"{name}_s{seed}.npz", p_binary=pb, p_multi=pm)
        save_cm(name, seed, pm)
        append_row(XGB_RESULTS, {"model": "xgboost", "variant": name, "seed": seed, "n_train": len(arr[3]),
                                 "best_val_score": val_score(VAL_ARR[2], VAL_ARR[3], vpb, vpm), **m})
        print(f"  {name:20s} s{seed}: bin_acc={m['bin_acc']:.4f} multi_acc={m['multi_acc']:.4f} "
              f"macroF1={m['multi_macro_f1']:.4f}")


[xgboost real_only seed0] 0.0 min
  xgb-real_only        s0: bin_acc=0.8612 multi_acc=0.7397 macroF1=0.4523
[xgboost ctgan_nofilter seed0] 0.1 min
  xgb-ctgan_nofilter   s0: bin_acc=0.8654 multi_acc=0.7391 macroF1=0.4746
[xgboost ctgan_shap seed0] 0.1 min
  xgb-ctgan_shap       s0: bin_acc=0.8611 multi_acc=0.7398 macroF1=0.4793
[xgboost ctgan_random seed0] 0.1 min
  xgb-ctgan_random     s0: bin_acc=0.8637 multi_acc=0.7394 macroF1=0.4791
[xgboost ctganD_nofilter seed0] 0.1 min
  xgb-ctganD_nofilter  s0: bin_acc=0.8627 multi_acc=0.7361 macroF1=0.4617
[xgboost ctganD_shap seed0] 0.1 min
  xgb-ctganD_shap      s0: bin_acc=0.8608 multi_acc=0.7367 macroF1=0.4651
[xgboost ctganD_random seed0] 0.1 min
  xgb-ctganD_random    s0: bin_acc=0.8609 multi_acc=0.7351 macroF1=0.4606
[xgboost tvae_nofilter seed0] 0.1 min
  xgb-tvae_nofilter    s0: bin_acc=0.8620 multi_acc=0.7378 macroF1=0.4558
[xgboost tvae_shap seed0] 0.1 min
  xgb-tvae_shap        s0: bin_acc=0.8640 multi_acc=0.7355 macroF1=0.4473
[xg

In [17]:
# %% Cell 13 — sensitivity of target ratio x removal fraction (VALIDATION ONLY; R2-3)
SENS = OUTPUT_DIR / "tables" / "sensitivity_validation.csv"
if RUN_SENSITIVITY and RUN_STAGE1:
    seed = SEEDS[0]
    done = set(zip(*[pd.read_csv(SENS)[c] for c in ["ratio", "removal"]])) if SENS.exists() else set()
    for ratio in SENS_RATIOS:
        for removal in SENS_REMOVALS:
            if (ratio, removal) in done:
                continue
            with Timer(f"sensitivity ratio={ratio} removal={removal}"):
                arr, _ = build_training("ctgan_shap", seed, removal=removal, ratio=ratio)
                model, hist, best_val = train_model(arr, VAL_ARR, seed)
                pb, pm = predict(model, *to_dev(VAL_ARR[0], VAL_ARR[1]))
            mv = {**binary_metrics(VAL_ARR[2], pb), **multi_metrics(VAL_ARR[3], pm)}
            append_row(SENS, {"ratio": ratio, "removal": removal, "seed": seed, "n_train": len(arr[3]),
                              "val_score": best_val, "val_bin_f1": mv["bin_f1_attack"],
                              "val_multi_macro_f1": mv["multi_macro_f1"], "val_multi_acc": mv["multi_acc"],
                              **{f"val_F1_{c}": mv[f"F1_{c}"] for c in MINORITY}})
            del model


[sensitivity ratio=0.4 removal=0.0] 0.0 min
[sensitivity ratio=0.4 removal=0.1] 0.0 min


In [18]:
# %% Cell 14 — realism, memorisation and coverage diagnostics for every generator (R2-4)
def nn_dist(ref, query):
    return NearestNeighbors(n_neighbors=1).fit(ref).kneighbors(query)[0][:, 0]


def coverage(ref, syn, k=5):
    radii = NearestNeighbors(n_neighbors=k + 1).fit(ref).kneighbors(ref)[0][:, -1]
    return float((NearestNeighbors(n_neighbors=1).fit(syn).kneighbors(ref)[0][:, 0] <= radii).mean())


def emb(df):
    return np.concatenate([prep.num(df), prep.onehot(prep.cat(df))], 1)


def sub(df, n=None):
    n = n or DIAG_MAX
    return df if len(df) <= n else df.sample(n, random_state=0)


DIAG = OUTPUT_DIR / "tables" / "synthetic_diagnostics.csv"
if RUN_DIAGNOSTICS:
    rows = []
    for gen in GENERATORS:
        for seed in SEEDS:
            scored, info = score_synthetic(gen, seed)
            kept = {m: select(scored, m, REMOVAL_FRAC, seed) for m in ["shap", "conf", "random"]}
            for cls in MINORITY:
                R = emb(sub(train_df[train_df.attack_cat == cls]))
                base = nn_dist(R, emb(val_df[val_df.attack_cat == cls]))
                thr = np.percentile(base, 5)
                sets = {"all": scored, "shap_kept": kept["shap"], "shap_removed": scored.drop(index=kept["shap"].index),
                        "conf_kept": kept["conf"], "random_kept": kept["random"]}
                for name, s in sets.items():
                    s = s[s.attack_cat == cls]
                    if len(s) == 0:
                        continue
                    S = emb(sub(s))
                    d = nn_dist(R, S)
                    rows.append({"generator": gen, "seed": seed, "class": cls, "set": name, "n": len(s),
                                 "median_DCR": float(np.median(d)), "median_DCR_real_holdout": float(np.median(base)),
                                 "dcr_ratio": float(np.median(d) / max(np.median(base), 1e-9)),
                                 "pct_closer_than_5pct_real": float((d < thr).mean() * 100),
                                 "pct_exact_copies": float((d < 1e-6).mean() * 100),
                                 "coverage_k5": coverage(R, S), "mean_p_synthetic": float(s["p_synthetic"].mean())})
    diag_df = pd.DataFrame(rows)
    diag_df.to_csv(DIAG, index=False)
    print(diag_df[diag_df.set == "all"].groupby("generator")[["dcr_ratio", "coverage_k5", "mean_p_synthetic"]]
          .mean().round(3))


           dcr_ratio  coverage_k5  mean_p_synthetic
generator                                          
ctgan          7.736        0.049             0.943
ctganD         7.582        0.067             0.928
tvae           6.509        0.107             0.939


In [19]:
# %% Cell 15 — correlation and statistical similarity (R1-7, reproducible definition)
from scipy.stats import ks_2samp


def realism_stats(real, syn):
    """Pearson correlation on the N_CORR_FEATURES highest-variance numerical features of the real data.
    mean_abs_diff: mean |r_real - r_syn| over the upper triangle (diagonal excluded), m(m-1)/2 pairs.
    frobenius: ||R_real - R_syn||_F over the full m x m matrix (diagonal differences are 0).
    similarity = (1 - mean_abs_diff / 2) * 100. m is reported so the three numbers can be checked:
    frobenius >= sqrt(2 * pairs) * mean_abs_diff for any m x m symmetric difference matrix."""
    feats = real[NUM_COLS].var().sort_values(ascending=False).head(N_CORR_FEATURES).index.tolist()
    rr = real[feats].corr().fillna(0).to_numpy()
    rs = syn[feats].corr().fillna(0).to_numpy()
    iu = np.triu_indices(len(feats), 1)
    mad = float(np.abs(rr[iu] - rs[iu]).mean())
    ks = [ks_2samp(real[c], syn[c]).statistic for c in NUM_COLS]
    rme = [abs(syn[c].mean() - real[c].mean()) / (abs(real[c].mean()) + 1e-9) * 100 for c in NUM_COLS]
    return {"m_features": len(feats), "pairs": len(iu[0]), "mean_abs_corr_diff": mad,
            "frobenius_diff": float(np.linalg.norm(rr - rs)), "corr_similarity_pct": (1 - mad / 2) * 100,
            "mean_ks_statistic": float(np.mean(ks)), "median_relative_mean_error_pct": float(np.median(rme)),
            "mean_relative_mean_error_pct": float(np.mean(rme)), "features": ";".join(feats)}


STATS = OUTPUT_DIR / "tables" / "realism_statistics.csv"
if RUN_STATS:
    rows = []
    for gen in GENERATORS:
        for seed in SEEDS:
            scored, _ = score_synthetic(gen, seed)
            kept = select(scored, "shap", REMOVAL_FRAC, seed)
            real_min = train_df[train_df.attack_cat.isin(MINORITY)]
            for scope in ["pooled"] + MINORITY:
                r = real_min if scope == "pooled" else real_min[real_min.attack_cat == scope]
                for name, s in [("all", scored), ("shap_kept", kept)]:
                    ss = s if scope == "pooled" else s[s.attack_cat == scope]
                    if len(ss) > 2 and len(r) > 2:
                        rows.append({"generator": gen, "seed": seed, "scope": scope, "set": name,
                                     **realism_stats(r, ss)})
    stats_df = pd.DataFrame(rows)
    stats_df.to_csv(STATS, index=False)
    print(stats_df[stats_df.scope == "pooled"].groupby(["generator", "set"])[
        ["corr_similarity_pct", "mean_abs_corr_diff", "frobenius_diff", "mean_ks_statistic"]].mean().round(4))


                     corr_similarity_pct  mean_abs_corr_diff  frobenius_diff  \
generator set                                                                  
ctgan     all                    89.6009              0.2080          6.0349   
          shap_kept              89.5895              0.2082          6.0385   
ctganD    all                    89.6947              0.2061          6.2808   
          shap_kept              89.6756              0.2065          6.2848   
tvae      all                    89.7153              0.2057          6.1644   
          shap_kept              89.7103              0.2058          6.1791   

                     mean_ks_statistic  
generator set                           
ctgan     all                   0.2545  
          shap_kept             0.2511  
ctganD    all                   0.2824  
          shap_kept             0.2774  
tvae      all                   0.2497  
          shap_kept             0.2462  


In [20]:
# %% Cell 16 — SHAP for the final intrusion classifiers + fidelity + stability (R2-5)
if RUN_EXPLAIN:
    import shap
    from scipy.stats import spearmanr
    FEAT_NAMES = NUM_COLS + CAT_COLS
    nnum = len(NUM_COLS)
    rng = np.random.default_rng(0)
    test_pick = np.concatenate([rng.choice(np.where(TEST_ARR[3] == i)[0],
                                           min(EXPLAIN_N // len(CLASSES), (TEST_ARR[3] == i).sum()), replace=False)
                                for i in range(len(CLASSES))])
    bg_pick = rng.choice(len(TRAIN_REAL[3]), EXPLAIN_BG, replace=False)
    Xexp = np.concatenate([TEST_ARR[0][test_pick], TEST_ARR[1][test_pick]], 1).astype(np.float64)
    Xbg = np.concatenate([TRAIN_REAL[0][bg_pick], TRAIN_REAL[1][bg_pick]], 1).astype(np.float64)
    fill = np.r_[np.median(Xbg[:, :nnum], 0), pd.DataFrame(Xbg[:, nnum:]).mode().iloc[0].to_numpy()]
    imp_rows, fid_rows, stab = [], [], []
    for variant in EXPLAIN_VARIANTS:
        for seed in SEEDS:
            path = OUTPUT_DIR / "models" / f"{variant}_s{seed}.pt"
            if not path.exists():
                continue
            model = FTTransformer(nnum, prep.cards).to(DEVICE)
            model.load_state_dict(torch.load(path, map_location=DEVICE))

            def f(X):
                xn = torch.as_tensor(X[:, :nnum], dtype=torch.float32, device=DEVICE)
                xc = torch.as_tensor(np.rint(X[:, nnum:]), dtype=torch.long, device=DEVICE)
                pb, pm = predict(model, xn, xc)
                return np.c_[pb, pm]

            with Timer(f"classifier SHAP {variant} seed{seed}"):
                ex = shap.PermutationExplainer(f, shap.maskers.Independent(Xbg, max_samples=EXPLAIN_BG), seed=seed)
                sv = ex(Xexp, max_evals=2 * Xexp.shape[1] + 1 if RUN_MODE == "smoke" else 500).values
            base_out = f(Xexp)
            pred_cls = base_out[:, 1:].argmax(1)
            idx = np.arange(len(Xexp))
            for name, phi, out_idx in [("binary_P_attack", sv[:, :, 0], np.zeros(len(Xexp), int)),
                                       ("multi_predicted_class", sv[idx, :, 1 + pred_cls], 1 + pred_cls)]:
                imp = np.abs(phi).mean(0)
                imp_rows += [{"variant": variant, "seed": seed, "output": name, "feature": FEAT_NAMES[j],
                              "mean_abs_shap": imp[j]} for j in range(len(FEAT_NAMES))]
                for k in [1, 3, 5, 10]:
                    Xt = Xexp.copy()
                    top = np.argsort(-np.abs(phi), 1)[:, :k]
                    for i in idx:
                        Xt[i, top[i]] = fill[top[i]]
                    drop_top = base_out[idx, out_idx] - f(Xt)[idx, out_idx]
                    drops_r = []
                    for _ in range(5):
                        Xr_ = Xexp.copy()
                        for i in idx:
                            cols = rng.choice(Xexp.shape[1], k, replace=False); Xr_[i, cols] = fill[cols]
                        drops_r.append(base_out[idx, out_idx] - f(Xr_)[idx, out_idx])
                    fid_rows.append({"variant": variant, "seed": seed, "output": name, "k": k,
                                     "mean_drop_top_shap": drop_top.mean(), "mean_drop_random": np.mean(drops_r)})
            del model
    imp_df, fid_df = pd.DataFrame(imp_rows), pd.DataFrame(fid_rows)
    imp_df.to_csv(OUTPUT_DIR / "tables" / "classifier_shap_importance.csv", index=False)
    fid_df.to_csv(OUTPUT_DIR / "tables" / "classifier_shap_fidelity.csv", index=False)
    if len(imp_df):
        for (variant, out_name), g in imp_df.groupby(["variant", "output"]):
            piv = g.pivot(index="feature", columns="seed", values="mean_abs_shap")
            for a in piv.columns:
                for b in piv.columns:
                    if a < b:
                        stab.append({"variant": variant, "output": out_name, "seed_a": a, "seed_b": b,
                                     "spearman": spearmanr(piv[a], piv[b]).correlation,
                                     "top10_overlap": len(set(piv[a].nlargest(10).index) &
                                                          set(piv[b].nlargest(10).index)) / 10})
        pd.DataFrame(stab).to_csv(OUTPUT_DIR / "tables" / "classifier_shap_stability.csv", index=False)
        print(fid_df.groupby(["variant", "output", "k"])[["mean_drop_top_shap", "mean_drop_random"]].mean().round(4))


[classifier SHAP ctgan_shap seed0] 0.0 min
[classifier SHAP ctgan_shap seed1] 0.0 min
[classifier SHAP real_only seed0] 0.0 min
[classifier SHAP real_only seed1] 0.0 min
                                     mean_drop_top_shap  mean_drop_random
variant    output                k                                       
ctgan_shap binary_P_attack       1               0.0006           -0.0015
                                 3               0.0098            0.0031
                                 5               0.0036           -0.0167
                                 10             -0.0601           -0.0291
           multi_predicted_class 1               0.0464            0.0059
                                 3               0.1299            0.0041
                                 5               0.1507            0.0213
                                 10              0.2332            0.0380
real_only  binary_P_attack       1               0.0295            0.0001
                

In [21]:
# %% Cell 17 — aggregate tables: mean ± sd, paired differences with 95% CI, McNemar (R2-3)
from scipy import stats

res = pd.concat([pd.read_csv(p) for p in [RESULTS, XGB_RESULTS] if p.exists()], ignore_index=True)
KEYS = [k for k in ["bin_acc", "bin_precision", "bin_recall", "bin_specificity", "bin_f1_attack", "bin_mcc",
                    "bin_roc_auc", "bin_pr_auc", "bin_gmean", "multi_acc", "multi_macro_precision",
                    "multi_macro_recall", "multi_macro_f1", "multi_weighted_f1", "multi_macro_roc_auc",
                    "novel_acc", "novel_macro_f1"] if k in res]
summary = res.groupby("variant")[KEYS].agg(["mean", "std"])
summary.columns = [f"{a}_{b}" for a, b in summary.columns]
summary.to_csv(OUTPUT_DIR / "tables" / "paper_results_mean_sd.csv")
fmt = pd.DataFrame({k: res.groupby("variant")[k].apply(
    lambda s: f"{100*s.mean():.2f} ± {100*s.std(ddof=1):.2f}" if len(s) > 1 else f"{100*s.mean():.2f}") for k in KEYS})
fmt.to_csv(OUTPUT_DIR / "tables" / "paper_results_formatted.csv")
print(fmt[["bin_acc", "bin_f1_attack", "multi_acc", "multi_macro_f1"]].to_string())
(res.groupby("variant")[[f"F1_{c}" for c in CLASSES]].mean() * 100).round(2) \
    .to_csv(OUTPUT_DIR / "tables" / "paper_per_class_f1.csv")


def paired(a_name, b_name, metric="multi_macro_f1"):
    a = res[res.variant == a_name].set_index("seed")
    b = res[res.variant == b_name].set_index("seed")
    common = a.index.intersection(b.index)
    if len(common) < 2 or metric not in a:
        return None
    d = (a.loc[common, metric] - b.loc[common, metric]).to_numpy() * 100
    n = len(d)
    ci = stats.t.ppf(0.975, n - 1) * d.std(ddof=1) / np.sqrt(n)
    return {"comparison": f"{a_name} - {b_name}", "metric": metric, "n_seeds": n, "mean_diff_pp": d.mean(),
            "ci95_low": d.mean() - ci, "ci95_high": d.mean() + ci,
            "wilcoxon_p": stats.wilcoxon(d).pvalue if n >= 5 and np.any(d != 0) else np.nan}


PAIRS = [("ctgan_shap", v) for v in STAGE1_VARIANTS if v != "ctgan_shap"] + [("ctgan_shap", "xgb-real_only")]
for g in ["ctganD", "tvae"]:
    PAIRS += [(f"{g}_shap", "real_only"), (f"{g}_shap", f"{g}_random"), (f"{g}_shap", f"{g}_nofilter"),
              (f"{g}_shap", "ctgan_shap")]
for v in XGB_STAGE2:
    PAIRS.append((f"xgb-{v}", "xgb-real_only"))
for g in ["ctgan", "ctganD", "tvae"]:
    PAIRS.append((f"xgb-{g}_shap", f"xgb-{g}_nofilter"))
metrics_for_pairs = ["multi_macro_f1", "multi_acc", "bin_f1_attack"] + [f"F1_{c}" for c in MINORITY]
pair_rows = [r for a, b in PAIRS for mt in metrics_for_pairs if (r := paired(a, b, mt))]
pair_df = pd.DataFrame(pair_rows)
pair_df.to_csv(OUTPUT_DIR / "tables" / "paper_paired_differences.csv", index=False)
if len(pair_df):
    print(pair_df[pair_df.metric == "multi_macro_f1"].round(3).to_string(index=False))

y_true = np.load(OUTPUT_DIR / "preds" / "test_labels.npz")["y_multi"]
mc = []
for a, b in PAIRS:
    for seed in SEEDS:
        pa, pbp = OUTPUT_DIR / "preds" / f"{a}_s{seed}.npz", OUTPUT_DIR / "preds" / f"{b}_s{seed}.npz"
        if pa.exists() and pbp.exists():
            ca = np.load(pa)["p_multi"].argmax(1) == y_true
            cb = np.load(pbp)["p_multi"].argmax(1) == y_true
            n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
            mc.append({"comparison": f"{a} vs {b}", "seed": seed, "a_only_correct": n01, "b_only_correct": n10,
                       "mcnemar_exact_p": stats.binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0})
pd.DataFrame(mc).to_csv(OUTPUT_DIR / "tables" / "paper_mcnemar.csv", index=False)


                          bin_acc bin_f1_attack      multi_acc multi_macro_f1
variant                                                                      
adasyn               80.38 ± 0.88  84.73 ± 0.51   42.31 ± 0.06   20.93 ± 1.31
ctganD_nofilter      85.29 ± 0.78  87.91 ± 0.56   48.08 ± 2.13   17.72 ± 0.57
ctganD_random        81.99 ± 3.46  85.71 ± 2.01   44.13 ± 3.99   16.55 ± 0.86
ctganD_shap          79.23 ± 2.32  84.11 ± 1.49   39.44 ± 1.78   16.07 ± 0.01
ctgan_conf           80.70 ± 1.64  84.98 ± 0.95   45.22 ± 0.78   21.95 ± 2.57
ctgan_nofilter       84.39 ± 2.30  87.27 ± 1.36   48.62 ± 0.08   22.51 ± 5.18
ctgan_random         82.34 ± 3.74  86.08 ± 2.34   43.39 ± 6.97   21.90 ± 6.21
ctgan_shap           82.15 ± 0.87  85.95 ± 0.53   47.20 ± 1.66   22.39 ± 1.91
ctgan_shaponly       83.60 ± 0.51  86.81 ± 0.23   46.78 ± 0.69   21.10 ± 2.53
real_only            81.16 ± 1.19  85.24 ± 0.75  57.23 ± 12.07   19.33 ± 5.07
smote                84.45 ± 0.14  87.34 ± 0.04   44.61 ± 0.16  

In [22]:
# %% Cell 18 — Stage 2 verdict against the pre-registered criteria
VERDICT = {}
if RUN_STAGE2:
    disc = pd.read_csv(OUTPUT_DIR / "tables" / "discriminator_auc.csv")
    diag = pd.read_csv(DIAG) if DIAG.exists() else pd.DataFrame()
    for model_kind, prefix in [("transformer", ""), ("xgboost", "xgb-")]:
        cands = {}
        for g in GENERATORS:
            v = f"{prefix}{g}_shap"
            if v in set(res.variant):
                cands[g] = res[res.variant == v]["best_val_score"].mean()
        if not cands:
            continue
        best = max(cands, key=cands.get)                       # selected on VALIDATION, not test
        auc = disc[disc.generator == best]["oof_auc_all"].mean()
        dcr = diag[(diag.generator == best) & (diag.set == "shap_kept")]["dcr_ratio"].mean() if len(diag) else np.nan
        c3 = {}
        for ref in CRITERIA["require_ci_above_zero_vs"]:
            ref_name = f"{prefix}real_only" if ref == "real_only" else f"{prefix}{best}_{ref}"
            r = paired(f"{prefix}{best}_shap", ref_name, CRITERIA["metric"])
            c3[ref_name] = r
        passed = {"C1_discriminator_auc": bool(auc < CRITERIA["max_discriminator_auc"]),
                  "C2_dcr_ratio": bool(dcr < CRITERIA["max_dcr_ratio"]),
                  **{f"C3_beats_{k}": bool(v is not None and v["ci95_low"] > 0) for k, v in c3.items()}}
        VERDICT[model_kind] = {"selected_generator": best, "validation_scores": cands,
                               "discriminator_auc": auc, "dcr_ratio": dcr, "comparisons": c3, "passed": passed,
                               "all_passed": all(passed.values())}
    VERDICT["criteria"] = CRITERIA
    VERDICT["decision"] = ("Route A: method paper (criteria met)"
                           if any(v.get("all_passed") for k, v in VERDICT.items() if isinstance(v, dict)
                                  and "all_passed" in v)
                           else "Route B: controlled evaluation study (criteria not met)")
    save_json(VERDICT, OUTPUT_DIR / "tables" / "stage2_verdict.json")
    print(json.dumps({k: (v["passed"] if isinstance(v, dict) and "passed" in v else v)
                      for k, v in VERDICT.items() if k != "criteria"}, indent=2, default=str))


{
  "transformer": {
    "C1_discriminator_auc": false,
    "C2_dcr_ratio": false,
    "C3_beats_real_only": false,
    "C3_beats_tvae_random": false
  },
  "xgboost": {
    "C1_discriminator_auc": false,
    "C2_dcr_ratio": false,
    "C3_beats_xgb-real_only": false,
    "C3_beats_xgb-ctgan_random": false
  },
  "decision": "Route B: controlled evaluation study (criteria not met)"
}


In [23]:
# %% Cell 19 — paper figures from the saved results (R1-8: every figure is rebuilt from files)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as sk_auc

FIG = OUTPUT_DIR / "figures"
plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.titlesize": 11, "figure.dpi": 110})


def savefig(fig, name):
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("  figure:", name)


def safe(fn):
    try:
        fn()
    except Exception as err:  # a missing optional table must not stop the remaining figures
        print(f"  skipped {fn.__name__}: {err}")


def fig_class_distribution():
    led = pd.read_csv(OUTPUT_DIR / "tables" / "ledger_counts.csv").set_index("stage")
    stages = [s for s in ["official_train", "train_after_dedup", "train_split", f"train_set_ctgan_shap_seed{SEEDS[0]}"]
              if s in led.index]
    fig, ax = plt.subplots(figsize=(10, 4.5))
    w = 0.8 / len(stages)
    for i, s in enumerate(stages):
        ax.bar(np.arange(len(CLASSES)) + i * w, led.loc[s, CLASSES].astype(float), w, label=s.replace("_", " "))
    ax.set_yscale("log"); ax.set_xticks(np.arange(len(CLASSES)) + w * (len(stages) - 1) / 2)
    ax.set_xticklabels(CLASSES, rotation=30, ha="right"); ax.set_ylabel("Records (log scale)")
    ax.set_title("Class counts at each pipeline stage"); ax.legend(fontsize=8)
    savefig(fig, "fig_class_distribution_by_stage")


def fig_training_curves():
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for v in ["real_only", "ctgan_shap", "ctganD_shap"]:
        p = OUTPUT_DIR / "logs" / f"history_{v}_s{SEEDS[0]}.csv"
        if p.exists():
            h = pd.read_csv(p)
            axes[0].plot(h.epoch, h.train_loss, marker="o", ms=3, label=v)
            axes[1].plot(h.epoch, h.val_score, marker="o", ms=3, label=v)
    axes[0].set_title("Training loss"); axes[1].set_title("Validation score (0.5 binary F1 + 0.5 macro-F1)")
    for a in axes:
        a.set_xlabel("Epoch"); a.legend(fontsize=8); a.grid(alpha=0.3)
    savefig(fig, "fig_training_curves")


def fig_confusion_matrices():
    names = [n for n in ["real_only", "ctgan_shap", "xgb-real_only"]
             if (OUTPUT_DIR / "tables" / f"cm_{n}_s{SEEDS[0]}.csv").exists()]
    fig, axes = plt.subplots(1, len(names), figsize=(5.2 * len(names), 4.6))
    axes = np.atleast_1d(axes)
    for ax, n in zip(axes, names):
        cms = [pd.read_csv(OUTPUT_DIR / "tables" / f"cm_{n}_s{s}.csv", index_col=0).to_numpy()
               for s in SEEDS if (OUTPUT_DIR / "tables" / f"cm_{n}_s{s}.csv").exists()]
        cm = np.mean(cms, 0)
        cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
        ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        for i in range(len(CLASSES)):
            for j in range(len(CLASSES)):
                ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if cmn[i, j] > 0.6 else "black")
        ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
        ax.set_xticklabels(CLASSES, rotation=60, ha="right", fontsize=7); ax.set_yticklabels(CLASSES, fontsize=7)
        ax.set_title(f"{n} (row-normalised, mean of {len(cms)} seeds)"); ax.set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    savefig(fig, "fig_confusion_matrices")


def fig_roc_binary():
    yb = np.load(OUTPUT_DIR / "preds" / "test_labels.npz")["y_binary"]
    fig, ax = plt.subplots(figsize=(5, 4.5))
    for n in ["real_only", "ctgan_shap", "ctgan_random", "smote", "xgb-real_only", "ctganD_shap", "tvae_shap"]:
        p = OUTPUT_DIR / "preds" / f"{n}_s{SEEDS[0]}.npz"
        if p.exists():
            pb = np.load(p)["p_binary"]
            fpr, tpr, _ = roc_curve(yb, pb)
            ax.plot(fpr, tpr, lw=1.2, label=f"{n} (AUC={sk_auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(f"Binary ROC, official test set (seed {SEEDS[0]})"); ax.legend(fontsize=7)
    savefig(fig, "fig_roc_binary")


def fig_roc_multiclass():
    ym = np.load(OUTPUT_DIR / "preds" / "test_labels.npz")["y_multi"]
    pm = np.load(OUTPUT_DIR / "preds" / f"ctgan_shap_s{SEEDS[0]}.npz")["p_multi"]
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    for i, c in enumerate(CLASSES):
        if (ym == i).any():
            fpr, tpr, _ = roc_curve(ym == i, pm[:, i])
            ax.plot(fpr, tpr, lw=1, label=f"{c} ({sk_auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title("One-vs-rest ROC, ctgan_shap"); ax.legend(fontsize=7)
    savefig(fig, "fig_roc_multiclass")


def fig_main_bars():
    g = res.groupby("variant")
    order = g["multi_macro_f1"].mean().sort_values().index
    fig, axes = plt.subplots(1, 2, figsize=(11, 0.35 * len(order) + 1.5))
    for ax, k, t in [(axes[0], "bin_f1_attack", "Binary F1 (attack), %"), (axes[1], "multi_macro_f1", "Macro-F1, %")]:
        m, s = g[k].mean()[order] * 100, g[k].std()[order].fillna(0) * 100
        ax.barh(range(len(order)), m, xerr=s, color=["#c44e52" if "xgb" in v else "#4c72b0" for v in order])
        ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=8); ax.set_title(t)
        ax.set_xlim(m.min() - 5, m.max() + 3); ax.grid(axis="x", alpha=0.3)
    savefig(fig, "fig_main_results_mean_sd")


def fig_per_class_heatmap():
    pc = pd.read_csv(OUTPUT_DIR / "tables" / "paper_per_class_f1.csv", index_col=0)
    fig, ax = plt.subplots(figsize=(10, 0.4 * len(pc) + 1.5))
    ax.imshow(pc.to_numpy(), cmap="viridis", aspect="auto", vmin=0, vmax=100)
    for i in range(pc.shape[0]):
        for j in range(pc.shape[1]):
            ax.text(j, i, f"{pc.iat[i, j]:.1f}", ha="center", va="center", fontsize=7,
                    color="white" if pc.iat[i, j] < 60 else "black")
    ax.set_xticks(range(pc.shape[1])); ax.set_xticklabels([c.replace("F1_", "") for c in pc.columns], rotation=30)
    ax.set_yticks(range(pc.shape[0])); ax.set_yticklabels(pc.index, fontsize=8)
    ax.set_title("Per-class F1 (%), mean over seeds, official test set")
    savefig(fig, "fig_per_class_f1")


def fig_sensitivity():
    s = pd.read_csv(SENS)
    piv = s.pivot(index="ratio", columns="removal", values="val_multi_macro_f1") * 100
    fig, ax = plt.subplots(figsize=(4.8, 3.6))
    ax.imshow(piv.to_numpy(), cmap="magma")
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, f"{piv.iat[i, j]:.1f}", ha="center", va="center", color="white")
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns); ax.set_xlabel("Removal fraction")
    ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index); ax.set_ylabel("Target ratio")
    ax.set_title("Validation macro-F1 (%)")
    savefig(fig, "fig_sensitivity_validation")


def fig_realism():
    disc = pd.read_csv(OUTPUT_DIR / "tables" / "discriminator_auc.csv")
    diag = pd.read_csv(DIAG)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
    a = disc.groupby("generator")["oof_auc_all"].agg(["mean", "std"])
    axes[0].bar(a.index, a["mean"], yerr=a["std"].fillna(0)); axes[0].axhline(CRITERIA["max_discriminator_auc"],
                                                                               ls="--", c="k", lw=0.8)
    axes[0].set_ylim(0.5, 1.01); axes[0].set_title("Discriminator AUC (out-of-fold)")
    d = diag[diag.set == "all"].groupby(["class", "generator"])["dcr_ratio"].mean().unstack()
    d.plot.bar(ax=axes[1], rot=30); axes[1].axhline(CRITERIA["max_dcr_ratio"], ls="--", c="k", lw=0.8)
    axes[1].set_title("Distance to real / real-holdout distance"); axes[1].set_xlabel("")
    c = diag[diag.set == "all"].groupby(["class", "generator"])["coverage_k5"].mean().unstack()
    c.plot.bar(ax=axes[2], rot=30, legend=False); axes[2].set_title("Coverage (k = 5)"); axes[2].set_xlabel("")
    savefig(fig, "fig_synthetic_realism")


def fig_refinement():
    kc = pd.read_csv(OUTPUT_DIR / "tables" / f"refinement_counts_ctgan_seed{SEEDS[0]}.csv", index_col=0).drop("Total")
    fig, ax = plt.subplots(figsize=(7, 3.6))
    x = np.arange(len(kc))
    ax.bar(x - 0.2, kc["generated"], 0.4, label="Generated"); ax.bar(x + 0.2, kc["kept_shap"], 0.4, label="Kept (SHAP)")
    ax.set_xticks(x); ax.set_xticklabels(kc.index, rotation=30); ax.legend()
    ax.set_title(f"Synthetic records before / after SHAP filtering (seed {SEEDS[0]})")
    savefig(fig, "fig_refinement_counts")


def fig_classifier_shap():
    imp = pd.read_csv(OUTPUT_DIR / "tables" / "classifier_shap_importance.csv")
    imp = imp[imp.variant == EXPLAIN_VARIANTS[0]]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, out in zip(axes, ["binary_P_attack", "multi_predicted_class"]):
        g = imp[imp.output == out].groupby("feature")["mean_abs_shap"].agg(["mean", "std"]).nlargest(15, "mean")[::-1]
        ax.barh(g.index, g["mean"], xerr=g["std"].fillna(0)); ax.set_title(f"{out}: mean |SHAP| ± sd over seeds")
    savefig(fig, "fig_classifier_shap_importance")
    fid = pd.read_csv(OUTPUT_DIR / "tables" / "classifier_shap_fidelity.csv")
    fid = fid[fid.variant == EXPLAIN_VARIANTS[0]].groupby(["output", "k"])[["mean_drop_top_shap",
                                                                             "mean_drop_random"]].mean()
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
    for ax, out in zip(axes, ["binary_P_attack", "multi_predicted_class"]):
        f_ = fid.loc[out]
        ax.plot(f_.index, f_["mean_drop_top_shap"], "o-", label="Top-|SHAP| features removed")
        ax.plot(f_.index, f_["mean_drop_random"], "s--", label="Random features removed")
        ax.set_xlabel("k features replaced"); ax.set_ylabel("Drop in output"); ax.set_title(out); ax.legend(fontsize=8)
    savefig(fig, "fig_classifier_shap_fidelity")


def fig_discriminator_shap():
    info = json.load(open(OUTPUT_DIR / "scores" / f"ctgan_r{TARGET_RATIO}_seed{SEEDS[0]}.json"))
    s = pd.Series(info["global_importance_42_features"]).nlargest(15)[::-1]
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.barh(s.index, s.values); ax.set_title("Real-vs-synthetic discriminator: mean |SHAP| (log-odds)")
    savefig(fig, "fig_discriminator_shap_importance")


if RUN_FIGURES:
    for fn in [fig_class_distribution, fig_training_curves, fig_confusion_matrices, fig_roc_binary,
               fig_roc_multiclass, fig_main_bars, fig_per_class_heatmap, fig_sensitivity, fig_realism,
               fig_refinement, fig_classifier_shap, fig_discriminator_shap]:
        safe(fn)


  figure: fig_class_distribution_by_stage
  figure: fig_training_curves
  figure: fig_confusion_matrices
  figure: fig_roc_binary
  figure: fig_roc_multiclass
  figure: fig_main_results_mean_sd
  figure: fig_per_class_f1
  figure: fig_sensitivity_validation
  figure: fig_synthetic_realism
  figure: fig_refinement_counts
  figure: fig_classifier_shap_importance
  figure: fig_classifier_shap_fidelity
  figure: fig_discriminator_shap_importance


In [24]:
# %% Cell 20 — reviewer_evidence.md: every reviewer comment with the numbers from THIS run
def T(name):
    p = OUTPUT_DIR / "tables" / name
    return pd.read_csv(p, index_col=0) if p.exists() else None


def pf(x, d=2):
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{d}f}"


def pair_txt(a, b, metric="multi_macro_f1"):
    r = paired(a, b, metric)
    return "n/a" if r is None else (f"{r['mean_diff_pp']:+.2f} pp (95% CI {r['ci95_low']:+.2f} to "
                                    f"{r['ci95_high']:+.2f}; Wilcoxon p = {pf(r['wilcoxon_p'], 3)}; n = {r['n_seeds']})")


def mean_sd(v, k):
    s = res[res.variant == v][k] * 100
    return "n/a" if s.empty else f"{s.mean():.2f} ± {s.std(ddof=1):.2f}"


L = []
add = L.append
audit = json.load(open(OUTPUT_DIR / "tables" / "data_audit.json"))
led = pd.read_csv(OUTPUT_DIR / "tables" / "ledger_counts.csv").set_index("stage")
add(f"# Reviewer evidence — generated by the revision notebook ({RUN_MODE} run)\n")
add(f"Seeds: {SEEDS}. Test set: official UNSW-NB15 test partition ({audit['test_rows']:,} records, never used for "
    "fitting, selection or early stopping). All numbers below are read from the files in this folder.\n")

add("## Reviewer 1\n")
add("**R1-1 (training ledger).** Official training partition "
    f"{audit['train_rows_official']:,} → {audit['train_exact_duplicates_removed']:,} exact duplicates removed → "
    f"{audit['train_rows_after_dedup']:,} → stratified split ({int(VAL_FRAC*100)}% validation, seed {SPLIT_SEED}): "
    f"{int(led.loc['train_split','total']):,} train / {int(led.loc['validation_split','total']):,} validation. "
    "Preprocessing, generators and the discriminator are fitted on the training split only; validation is used only "
    "for early stopping and the sensitivity study; there is no refit on train+validation. "
    "Per-class counts at every stage: `tables/ledger_counts.csv`; split indices: `tables/split_*_rows.csv`.\n")
add(f"Additional audit: {audit['train_rows_with_same_features_different_class']:,} training records share identical "
    f"features with a record of another class; the test set has {audit['test_exact_duplicate_rows']:,} exact "
    f"duplicate rows and {audit['test_rows_whose_features_appear_in_train']:,} rows whose features occur in training "
    "(results on the remaining 'novel' test rows are reported as novel_*).\n")
kc = T(f"refinement_counts_ctgan_seed{SEEDS[0]}.csv")
if kc is not None:
    add(f"**R1-2 (removed records).** Seed {SEEDS[0]}, CTGAN, removal {int(REMOVAL_FRAC*100)}% per class:\n")
    add("| Class | Generated | Kept (SHAP) | Removed |\n|---|---|---|---|")
    for c, r in kc.iterrows():
        add(f"| {c} | {int(r['generated']):,} | {int(r['kept_shap']):,} | {int(r['generated']-r['kept_shap']):,} |")
    add("\nAll seeds: `tables/refinement_counts_<generator>_seed<k>.csv`; record-level masks: "
        "`tables/retention_masks_*.csv.gz`.\n")
add("**R1-3 (ablation F1).** Every metric is recomputed from saved predictions (`preds/*.npz`) and confusion "
    "matrices (`tables/cm_*.csv`). Binary metrics: positive class = Attack, threshold "
    f"{BINARY_THRESHOLD}. Multi-class: macro = unweighted mean over the 10 classes; weighted = support-weighted.\n")
add("**R1-4 (MCC).** MCC = (TP·TN − FP·FN) / √((TP+FP)(TP+FN)(TN+FP)(TN+FN)), computed with "
    "`sklearn.metrics.matthews_corrcoef`.\n")
add(f"**R1-5 (architecture).** Inputs: {len(NUM_COLS)} numerical features (signed log1p, z-score from the training "
    f"split, clipped to ±10) + {len(CAT_COLS)} categorical features (integer tokens, 0 = unseen; cardinalities "
    f"{prep.cards}). Each feature becomes one d = {D_MODEL} token (linear tokenizer for numerical, embedding for "
    f"categorical) plus one CLS token → sequence length {len(FEATURES)+1}. Encoder: {N_LAYERS} pre-norm layers, "
    f"{N_HEADS} heads, feed-forward {D_FF}, GELU, dropout {DROPOUT}; CLS pooling; two heads (binary sigmoid, "
    f"10-class softmax). Loss = BCE + {LAMBDA_MULTI}·CE; AdamW (lr {LR}, weight decay {WEIGHT_DECAY}), batch "
    f"{BATCH}, ≤ {MAX_EPOCHS} epochs, early stopping patience {PATIENCE} on validation (0.5·binary F1 + "
    "0.5·macro-F1), ReduceLROnPlateau (factor 0.5, patience 2), mixed precision. SHAP results are reported on the "
    f"{len(FEATURES)} original features (one-hot contributions summed per feature).\n")
add(f"**R1-6 (filtering rule).** Discriminator: XGBoost ({DISC_TREES} trees, depth 6, lr 0.05) on real minority "
    "training records vs synthetic records, features = scaled numerical + one-hot categorical + class indicator, "
    f"{DISC_FOLDS}-fold cross-fitting so every synthetic record is scored by a model that did not see it. SHAP: exact "
    "TreeSHAP on the log-odds output, positive = synthetic. Score q = 1 − [α·p_syn + (1−α)·D], D = Σmax(φ,0)/Σ|φ|, "
    f"α = {QUALITY_ALPHA}. Rule: within each class remove the {int(REMOVAL_FRAC*100)}% lowest q. The dataset-level "
    "quality assessment is descriptive only and removes no record.\n")
st = pd.read_csv(STATS) if STATS.exists() else None
if st is not None and len(st):
    s0 = st[(st.generator == "ctgan") & (st.scope == "pooled") & (st.set == "all")]
    if len(s0):
        add(f"**R1-7 (correlation summary).** m = {int(s0.m_features.iloc[0])} highest-variance numerical features "
            f"of the real minority training data ({int(s0.pairs.iloc[0])} pairs, Pearson, diagonal excluded from the "
            f"mean). CTGAN, mean over seeds: mean |Δr| = {s0.mean_abs_corr_diff.mean():.4f}, Frobenius "
            f"‖ΔR‖_F = {s0.frobenius_diff.mean():.4f} (full m×m matrix), similarity = (1 − mean|Δr|/2)·100 = "
            f"{s0.corr_similarity_pct.mean():.2f}%, mean KS = {s0.mean_ks_statistic.mean():.4f}. Per class and per "
            "generator: `tables/realism_statistics.csv`.\n")
add("**R1-8 (reproducibility).** This notebook + `run_config.json` (all settings, library versions, GPU), split "
    "indices, generator checkpoints for seed(s) " f"{SAVE_GENERATOR_SEEDS} (`generators.zip`), retention masks, "
    "Transformer weights (`models/`), test predictions for every run (`preds/`), and Cell 19, which rebuilds every "
    "figure from these files.\n")

add("## Reviewer 2\n")
add("**R2-1 (what SHAP adds).** Macro-F1, Transformer, matched removal count:\n")
for b in ["ctgan_random", "ctgan_conf", "ctgan_nofilter", "ctgan_shaponly", "real_only"]:
    add(f"- ctgan_shap − {b}: {pair_txt('ctgan_shap', b)}")
add("")
add("**R2-2 (baselines, identical splits).** Mean ± sd over seeds:\n")
add("| Variant | Binary acc. | Binary F1 | Multi acc. | Macro-F1 |\n|---|---|---|---|---|")
for v in [v for v in res.groupby("variant")["multi_macro_f1"].mean().sort_values(ascending=False).index]:
    add(f"| {v} | {mean_sd(v,'bin_acc')} | {mean_sd(v,'bin_f1_attack')} | {mean_sd(v,'multi_acc')} | "
        f"{mean_sd(v,'multi_macro_f1')} |")
add(f"\nctgan_shap − xgb-real_only (macro-F1): {pair_txt('ctgan_shap', 'xgb-real_only')}. "
    "IDS-MTran was not re-implemented; the FT-Transformer trained on real data is the Transformer baseline.\n")
add("**R2-3 (repeated runs, minority classes, sensitivity).** Per-class F1 changes, ctgan_shap − real_only:\n")
for c in MINORITY:
    add(f"- {c}: {pair_txt('ctgan_shap', 'real_only', f'F1_{c}')}")
if SENS.exists():
    s = pd.read_csv(SENS)
    b = s.loc[s.val_multi_macro_f1.idxmax()]
    add(f"\nSensitivity (validation only, seed {SEEDS[0]}): best validation macro-F1 "
        f"{b.val_multi_macro_f1*100:.2f}% at ratio {b.ratio}, removal {b.removal}; full grid in "
        "`tables/sensitivity_validation.csv`.\n")
disc = T("discriminator_auc.csv")
if DIAG.exists():
    dg = pd.read_csv(DIAG)
    add("**R2-4 (realism, memorisation, coverage).** Mean over seeds and classes:\n")
    add("| Generator | Discriminator AUC | DCR ratio (all) | DCR ratio (SHAP-kept) | Near-copies % | Coverage |\n"
        "|---|---|---|---|---|---|")
    dd = pd.read_csv(OUTPUT_DIR / "tables" / "discriminator_auc.csv")
    for g in GENERATORS:
        a = dg[(dg.generator == g) & (dg.set == "all")]
        k = dg[(dg.generator == g) & (dg.set == "shap_kept")]
        add(f"| {g} | {dd[dd.generator == g].oof_auc_all.mean():.4f} | {a.dcr_ratio.mean():.2f} | "
            f"{k.dcr_ratio.mean():.2f} | {a.pct_closer_than_5pct_real.mean():.2f} | {a.coverage_k5.mean():.4f} |")
    add("\nDomain violations before repair: `tables/generator_domain_violations.csv`.\n")
fid = T("classifier_shap_fidelity.csv")
stb = T("classifier_shap_stability.csv")
if fid is not None:
    fid = fid.reset_index()
    f5 = fid[(fid.variant == EXPLAIN_VARIANTS[0]) & (fid.k == 5)].groupby("output")[
        ["mean_drop_top_shap", "mean_drop_random"]].mean()
    add("**R2-5 (explanations of the final classifier).** Permutation SHAP on the trained Transformer "
        f"({EXPLAIN_VARIANTS[0]}), {EXPLAIN_N} stratified test records, {EXPLAIN_BG} background records. Deletion "
        "fidelity (k = 5): " + "; ".join(f"{o}: drop {r.mean_drop_top_shap:.3f} vs random {r.mean_drop_random:.3f}"
                                          for o, r in f5.iterrows()) + ".")
    if stb is not None:
        stb = stb.reset_index()
        sb = stb[stb.variant == EXPLAIN_VARIANTS[0]].groupby("output")[["spearman", "top10_overlap"]].mean()
        add(" Stability across seeds: " + "; ".join(f"{o}: Spearman {r.spearman:.3f}, top-10 overlap "
                                                     f"{r.top10_overlap:.2f}" for o, r in sb.iterrows()) + ".")
    add(" No cross-dataset or temporal evaluation was run; conclusions are restricted to UNSW-NB15.\n")
if VERDICT:
    add("## Stage 2 — pre-registered verdict\n")
    for k, v in VERDICT.items():
        if isinstance(v, dict) and "passed" in v:
            add(f"- **{k}**: selected generator `{v['selected_generator']}` (by validation); "
                + ", ".join(f"{c} = {'PASS' if ok else 'FAIL'}" for c, ok in v["passed"].items()))
    add(f"\n**Decision: {VERDICT['decision']}**\n")
(OUTPUT_DIR / "reviewer_evidence.md").write_text("\n".join(L), encoding="utf-8")
print("\n".join(L[:12]), "\n...\nWritten:", OUTPUT_DIR / "reviewer_evidence.md")


# Reviewer evidence — generated by the revision notebook (smoke run)

Seeds: [0, 1]. Test set: official UNSW-NB15 test partition (82,332 records, never used for fitting, selection or early stopping). All numbers below are read from the files in this folder.

## Reviewer 1

**R1-1 (training ledger).** Official training partition 175,341 → 67,601 exact duplicates removed → 107,740 → stratified split (20% validation, seed 42): 8,631 train / 2,158 validation. Preprocessing, generators and the discriminator are fitted on the training split only; validation is used only for early stopping and the sensitivity study; there is no refit on train+validation. Per-class counts at every stage: `tables/ledger_counts.csv`; split indices: `tables/split_*_rows.csv`.

Additional audit: 8,472 training records share identical features with a record of another class; the test set has 28,386 exact duplicate rows and 8,541 rows whose features occur in training (results on the remaining 'novel' test rows are r

In [25]:
# %% Cell 21 — configuration + downloads
import platform, zipfile
config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, tuple, bool, dict))
          and not k.startswith("_") and not k.endswith("_ARR") and k not in ("LEDGER", "TRAIN_REAL", "VERDICT",
                                                                              "SCORED")}
config.update(python=platform.python_version(), torch=torch.__version__, xgboost=xgb.__version__,
              pandas=pd.__version__, numpy=np.__version__, gpu=torch.cuda.get_device_name(0) if USE_AMP else "cpu")
try:
    import ctgan as _c; config["ctgan"] = _c.__version__
except Exception:
    pass
save_json(config, OUTPUT_DIR / "run_config.json")
main_zip = OUTPUT_DIR.parent / f"xct_ids_v2_{RUN_MODE}.zip"
with zipfile.ZipFile(main_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in OUTPUT_DIR.rglob("*"):
        if fp.is_file() and fp.parent.name not in ("synthetic", "scores", "generators"):
            z.write(fp, fp.relative_to(OUTPUT_DIR))
gen_zip = OUTPUT_DIR.parent / f"xct_ids_v2_{RUN_MODE}_generators.zip"
with zipfile.ZipFile(gen_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in (OUTPUT_DIR / "generators").glob("*"):
        z.write(fp, fp.name)
print(f"Download from the Output tab:\n  {main_zip} ({main_zip.stat().st_size/1e6:.1f} MB)  <- send this one"
      f"\n  {gen_zip} ({gen_zip.stat().st_size/1e6:.1f} MB)  <- generator checkpoints for the public repository")


Download from the Output tab:
  /kaggle/working/xct_ids_v2/xct_ids_v2_smoke.zip (31.2 MB)  <- send this one
  /kaggle/working/xct_ids_v2/xct_ids_v2_smoke_generators.zip (29.0 MB)  <- generator checkpoints for the public repository
